In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
from IPython.display import display
from src.db import query_df
from src.charts import monthly_trend, channel_bar, contribution_bar, risk_scatter

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_columns", 50)
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

## 1. 本期 vs 上期

In [2]:
display(query_df("SELECT * FROM `v_申请审批月度` ORDER BY `月份` DESC LIMIT 2"))

,月份,进件量,审批通过量,实际用信量,拒绝量,取消量,未使用额度量,审批通过率,用信率,审批金额,件均
0,2026-09,36012,8245.0,8070.0,6827.0,20940.0,175.0,0.2290,0.9788,188680.95,23.38
1,2026-08,28537,8394.0,7975.0,4292.0,15851.0,419.0,0.2941,0.9501,142868.64,17.91


## 2. 渠道贡献

In [3]:
decomp = query_df("SELECT * FROM `v_渠道归因` ORDER BY `总效应`")
display(decomp)

,渠道类型,基期占比,当期占比,基期通过率,当期通过率,结构效应,比率效应,总效应
0,Country-wide,0.1666,0.1217,0.9064,0.8764,-0.027462,-0.003652,-0.031114
1,Credit and cash offices,0.6026,0.6506,0.0856,0.0784,-0.010009,-0.004684,-0.014694
2,Stone,0.0547,0.0388,0.9219,0.8812,-0.010009,-0.001579,-0.011587
3,Regional / Local,0.0354,0.0299,0.9129,0.8744,-0.003429,-0.001149,-0.004578
4,AP+ (Cash loan),0.0085,0.0130,0.3909,0.2308,0.000434,-0.002081,-0.001647
5,Contact center,0.1246,0.1371,0.0309,0.0433,-0.003291,0.001701,-0.001591
6,Channel of corporate sales,0.0075,0.0089,0.2150,0.2288,-0.000108,0.000122,0.000015


## 3. 风险验证

In [4]:
channel = query_df("SELECT `渠道类型`,`进件量`,`审批通过率` FROM `v_渠道月度` WHERE `月份`='2026-09'")
risk = query_df("SELECT * FROM `v_还款风险`")
display(channel.merge(risk, on="渠道类型", how="left").sort_values("进件量", ascending=False))

,渠道类型,进件量,审批通过率,有还款记录合同数,DPD30合同数,DPD30率
4,Credit and cash offices,23430,0.0784,251520,3776,0.0150
2,Contact center,4939,0.0433,20144,966,0.0480
3,Country-wide,4384,0.8764,382101,11228,0.0294
6,Stone,1397,0.8812,184034,5440,0.0296
5,Regional / Local,1075,0.8744,91583,1736,0.0190
0,AP+ (Cash loan),468,0.2308,27292,503,0.0184
1,Channel of corporate sales,319,0.2288,1991,12,0.0060


## 本步结论

2026-09 审批通过率下降主要由渠道结构变化驱动，同时存在一定的渠道自身收紧。审批金额增长靠件均和期限，不靠件数；风险验证未发现“低通过必然低风险”的证据。建议先复核低通过率、高进件占比渠道，再检查审批规则与客群质量。